In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Project paths
RAW_DATA_PATH = Path("../data/raw")
PROCESSED_DATA_PATH = Path("../data/processed")

# Create processed-data folder if it doesn't exist
PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)

# Find all raw CSV files
csv_files = sorted(RAW_DATA_PATH.glob("*.csv"))

print("Number of raw CSV files:", len(csv_files))

for file in csv_files:
    print("-", file.name)

In [ ]:
# Load and combine all raw CSV files

dataframes = []

for file in csv_files:
    print("Loading:", file.name)

    df = pd.read_csv(file)

    # Remove extra spaces from column names
    df.columns = df.columns.str.strip()

    dataframes.append(df)

# Combine all files
combined_df = pd.concat(dataframes, ignore_index=True)

print("\n" + "=" * 70)
print("Combined dataset shape:", combined_df.shape)
print("Number of rows:", combined_df.shape[0])
print("Number of columns:", combined_df.shape[1])

In [ ]:
# Check missing values in the complete combined dataset

missing_values = combined_df.isnull().sum()

# Keep only columns that contain missing values
missing_values = missing_values[missing_values > 0]

print("Columns containing missing values:")
print(missing_values)

print("\n" + "=" * 70)
print("Total missing values:", missing_values.sum())

In [ ]:
# Remove rows containing missing values

rows_before = len(combined_df)

# Remove rows with any missing value
combined_df = combined_df.dropna()

rows_after = len(combined_df)

rows_removed = rows_before - rows_after

print("Rows before removing missing values:", rows_before)
print("Rows removed:", rows_removed)
print("Rows after removing missing values:", rows_after)

print("\nRemaining missing values:", combined_df.isnull().sum().sum())

In [ ]:
# Replace infinite values with NaN

positive_infinity = np.isposinf(combined_df.select_dtypes(include=np.number)).sum().sum()
negative_infinity = np.isneginf(combined_df.select_dtypes(include=np.number)).sum().sum()

print("Positive infinite values:", positive_infinity)
print("Negative infinite values:", negative_infinity)
print("Total infinite values:", positive_infinity + negative_infinity)

# Replace +inf and -inf with NaN
combined_df.replace([np.inf, -np.inf], np.nan, inplace=True)

print("\nInfinite values after replacement:",
      np.isinf(combined_df.select_dtypes(include=np.number)).sum().sum())

print("New missing values created:",
      combined_df.isnull().sum().sum())

In [ ]:
# Remove rows containing values that were originally infinite

rows_before = len(combined_df)

combined_df = combined_df.dropna()

rows_after = len(combined_df)

rows_removed = rows_before - rows_after

print("Rows before removing infinite-value rows:", rows_before)
print("Rows removed:", rows_removed)
print("Rows after removing infinite-value rows:", rows_after)

print("\nRemaining missing values:",
      combined_df.isnull().sum().sum())

print("Remaining infinite values:",
      np.isinf(
          combined_df.select_dtypes(include=np.number)
      ).sum().sum())

In [ ]:
# Remove duplicate rows

rows_before = len(combined_df)

# Remove exact duplicate rows
combined_df = combined_df.drop_duplicates()

rows_after = len(combined_df)

duplicates_removed = rows_before - rows_after

print("Rows before duplicate removal:", rows_before)
print("Duplicate rows removed:", duplicates_removed)
print("Rows after duplicate removal:", rows_after)

print("\nRemaining duplicate rows:",
      combined_df.duplicated().sum())

In [ ]:
# Check attack labels after data cleaning

label_counts = combined_df["Label"].value_counts()

print("Number of unique labels:", combined_df["Label"].nunique())

print("\nLabel distribution:")
print(label_counts)

In [ ]:
# Standardize attack label names

combined_df["Label"] = combined_df["Label"].str.strip()

label_mapping = {
    "DoS Hulk": "DoS_Hulk",
    "DoS GoldenEye": "DoS_GoldenEye",
    "DoS slowloris": "DoS_Slowloris",
    "DoS Slowhttptest": "DoS_Slowhttptest",
    "FTP-Patator": "FTP_Patator",
    "SSH-Patator": "SSH_Patator",
    "PortScan": "PortScan",
    "DDoS": "DDoS",
    "Bot": "Bot",
    "Web Attack � Brute Force": "WebAttack_BruteForce",
    "Web Attack � XSS": "WebAttack_XSS",
    "Web Attack � Sql Injection": "WebAttack_SQLInjection",
    "Infiltration": "Infiltration",
    "Heartbleed": "Heartbleed",
    "BENIGN": "BENIGN"
}

combined_df["Label"] = combined_df["Label"].replace(label_mapping)

print("Unique labels after standardization:")
print(combined_df["Label"].unique())

print("\nNumber of unique labels:",
      combined_df["Label"].nunique())

In [ ]:
# Create binary classification target
# BENIGN = normal traffic
# ATTACK = any non-BENIGN traffic

combined_df["Traffic_Type"] = combined_df["Label"].apply(
    lambda x: "BENIGN" if x == "BENIGN" else "ATTACK"
)

print("Binary traffic distribution:")
print(combined_df["Traffic_Type"].value_counts())

print("\nPercentage distribution:")
print(
    combined_df["Traffic_Type"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

In [ ]:
# Check data types of all columns

print("Data types of each column:")
print(combined_df.dtypes)

print("\n" + "="*70)
print("Number of numeric columns:",
      combined_df.select_dtypes(include=np.number).shape[1])

print("Number of non-numeric columns:",
      combined_df.select_dtypes(exclude=np.number).shape[1])

In [ ]:
# Separate input features and target variables

X = combined_df.drop(columns=["Label", "Traffic_Type"])
y_multiclass = combined_df["Label"]
y_binary = combined_df["Traffic_Type"]

print("Feature matrix shape:", X.shape)
print("Multiclass target shape:", y_multiclass.shape)
print("Binary target shape:", y_binary.shape)

print("\nNumber of feature columns:", X.shape[1])

print("\nTarget columns:")
print("- Multiclass target: Label")
print("- Binary target: Traffic_Type")

In [ ]:
# Display all feature columns

print("Number of feature columns:", len(X.columns))

print("\nFeature columns:")
for i, column in enumerate(X.columns, start=1):
    print(f"{i}. {column}")

In [ ]:
# Check for constant features in the complete cleaned dataset

constant_features = X.nunique()
constant_features = constant_features[constant_features <= 1]

print("Number of constant features:", len(constant_features))

if len(constant_features) > 0:
    print("\nConstant features:")
    for column in constant_features.index:
        print("-", column)
else:
    print("\nNo constant features found.")

In [ ]:
# Remove constant features

constant_feature_names = constant_features.index.tolist()

print("Removing constant features:")
for column in constant_feature_names:
    print("-", column)

X = X.drop(columns=constant_feature_names)

print("\nFeature matrix after removing constant features:", X.shape)
print("Number of remaining features:", X.shape[1])

In [ ]:
# Check the number of unique values in each remaining feature

feature_variability = X.nunique().sort_values()

print("Features with the fewest unique values:")
print(feature_variability.head(15))

print("\nFeatures with the most unique values:")
print(feature_variability.tail(10))

In [ ]:
# Check minimum and maximum values of each feature

feature_ranges = pd.DataFrame({
    "Minimum": X.min(),
    "Maximum": X.max(),
    "Mean": X.mean()
})

print("Feature ranges:")
display(feature_ranges)

In [ ]:
# Check for negative values in the feature dataset

negative_counts = (X < 0).sum()
negative_counts = negative_counts[negative_counts > 0].sort_values(ascending=False)

print("Features containing negative values:")
print(negative_counts)

print("\nNumber of features with negative values:",
      len(negative_counts))

In [ ]:
# Inspect the actual negative values

negative_features = negative_counts.index.tolist()

for column in negative_features:
    negative_values = X.loc[X[column] < 0, column]

    print("\n" + "="*70)
    print("FEATURE:", column)
    print("Negative records:", len(negative_values))
    print("Unique negative values:")
    print(sorted(negative_values.unique())[:20])

In [ ]:
# Features where negative values are physically invalid
invalid_negative_features = [
    "Flow IAT Min",
    "Flow Packets/s",
    "Flow Duration",
    "Flow IAT Mean",
    "Flow IAT Max",
    "Flow Bytes/s",
    "Fwd Header Length",
    "Fwd Header Length.1",
    "min_seg_size_forward",
    "Bwd Header Length",
    "Fwd IAT Min"
]

# Count rows having at least one invalid negative value
invalid_negative_mask = (X[invalid_negative_features] < 0).any(axis=1)

print("Rows containing invalid negative measurements:",
      invalid_negative_mask.sum())

print("Percentage of dataset:",
      round(invalid_negative_mask.mean() * 100, 4), "%")

print("\nRows not affected:",
      (~invalid_negative_mask).sum())

In [ ]:
rows_before = len(X)

# Remove rows containing physically invalid negative measurements
X = X.loc[~invalid_negative_mask].copy()

# Keep target values aligned with the remaining feature rows
y_multiclass = y_multiclass.loc[X.index].copy()
y_binary = y_binary.loc[X.index].copy()

rows_after = len(X)
rows_removed = rows_before - rows_after

print("Rows before removing invalid negatives:", rows_before)
print("Rows removed:", rows_removed)
print("Rows after removal:", rows_after)

print("\nFeature matrix shape:", X.shape)
print("Multiclass target shape:", y_multiclass.shape)
print("Binary target shape:", y_binary.shape)

In [ ]:
remaining_invalid_negatives = (X[invalid_negative_features] < 0).sum()

remaining_invalid_negatives = remaining_invalid_negatives[
    remaining_invalid_negatives > 0
]

print("Remaining invalid negative values:")

if len(remaining_invalid_negatives) == 0:
    print("None")
else:
    print(remaining_invalid_negatives)

print("\nTotal remaining invalid negative values:",
      remaining_invalid_negatives.sum())

In [ ]:
X = X.reset_index(drop=True)
y_multiclass = y_multiclass.reset_index(drop=True)
y_binary = y_binary.reset_index(drop=True)

print("Feature matrix shape:", X.shape)
print("Multiclass target shape:", y_multiclass.shape)
print("Binary target shape:", y_binary.shape)

print("\nFirst 5 X indices:", X.index[:5].tolist())
print("First 5 multiclass indices:", y_multiclass.index[:5].tolist())
print("First 5 binary indices:", y_binary.index[:5].tolist())

print("\nLast X index:", X.index[-1])
print("Expected last index:", len(X) - 1)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_binary_train, y_binary_test, y_multiclass_train, y_multiclass_test = train_test_split(
    X,
    y_binary,
    y_multiclass,
    test_size=0.20,
    random_state=42,
    stratify=y_multiclass
)

print("Training feature shape:", X_train.shape)
print("Testing feature shape:", X_test.shape)

print("\nBinary training target:", y_binary_train.shape)
print("Binary testing target:", y_binary_test.shape)

print("\nMulticlass training target:", y_multiclass_train.shape)
print("Multiclass testing target:", y_multiclass_test.shape)

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Binary target encoding
binary_encoder = LabelEncoder()

y_binary_train_encoded = binary_encoder.fit_transform(y_binary_train)
y_binary_test_encoded = binary_encoder.transform(y_binary_test)


# Multiclass target encoding
multiclass_encoder = LabelEncoder()

y_multiclass_train_encoded = multiclass_encoder.fit_transform(y_multiclass_train)
y_multiclass_test_encoded = multiclass_encoder.transform(y_multiclass_test)


# Display encoding mappings
print("Binary label mapping:")
for label, encoded_value in zip(
    binary_encoder.classes_,
    range(len(binary_encoder.classes_))
):
    print(f"{label} -> {encoded_value}")


print("\n" + "="*70)

print("Multiclass label mapping:")
for label, encoded_value in zip(
    multiclass_encoder.classes_,
    range(len(multiclass_encoder.classes_))
):
    print(f"{label} -> {encoded_value}")


print("\n" + "="*70)

print("Binary training shape:", y_binary_train_encoded.shape)
print("Binary testing shape:", y_binary_test_encoded.shape)

print("Multiclass training shape:", y_multiclass_train_encoded.shape)
print("Multiclass testing shape:", y_multiclass_test_encoded.shape)

In [ ]:
print("Binary Training Distribution:")
binary_train_distribution = pd.Series(
    y_binary_train_encoded
).value_counts().sort_index()

for encoded_value, count in binary_train_distribution.items():
    label = binary_encoder.inverse_transform([encoded_value])[0]
    percentage = count / len(y_binary_train_encoded) * 100
    print(f"{encoded_value} ({label}): {count:,} ({percentage:.2f}%)")


print("\n" + "="*70)

print("Binary Testing Distribution:")
binary_test_distribution = pd.Series(
    y_binary_test_encoded
).value_counts().sort_index()

for encoded_value, count in binary_test_distribution.items():
    label = binary_encoder.inverse_transform([encoded_value])[0]
    percentage = count / len(y_binary_test_encoded) * 100
    print(f"{encoded_value} ({label}): {count:,} ({percentage:.2f}%)")


print("\n" + "="*70)

print("Multiclass Training Distribution:")
multiclass_train_distribution = pd.Series(
    y_multiclass_train_encoded
).value_counts().sort_index()

for encoded_value, count in multiclass_train_distribution.items():
    label = multiclass_encoder.inverse_transform([encoded_value])[0]
    percentage = count / len(y_multiclass_train_encoded) * 100
    print(f"{encoded_value} ({label}): {count:,} ({percentage:.4f}%)")


print("\n" + "="*70)

print("Multiclass Testing Distribution:")
multiclass_test_distribution = pd.Series(
    y_multiclass_test_encoded
).value_counts().sort_index()

for encoded_value, count in multiclass_test_distribution.items():
    label = multiclass_encoder.inverse_transform([encoded_value])[0]
    percentage = count / len(y_multiclass_test_encoded) * 100
    print(f"{encoded_value} ({label}): {count:,} ({percentage:.4f}%)")

In [ ]:
from sklearn.preprocessing import StandardScaler

# Create the scaler
scaler = StandardScaler()

# Fit ONLY on training data and transform training data
X_train_scaled = scaler.fit_transform(X_train)

# Transform test data using the same scaler
X_test_scaled = scaler.transform(X_test)

print("Original training shape:", X_train.shape)
print("Original testing shape:", X_test.shape)

print("\nScaled training shape:", X_train_scaled.shape)
print("Scaled testing shape:", X_test_scaled.shape)

print("\nScaler fitted successfully.")

In [ ]:
# Calculate mean and standard deviation of scaled training data
train_means = X_train_scaled.mean(axis=0)
train_stds = X_train_scaled.std(axis=0)

print("Average absolute mean:",
      np.mean(np.abs(train_means)))

print("Average standard deviation:",
      np.mean(train_stds))

print("Maximum absolute mean:",
      np.max(np.abs(train_means)))

print("Minimum standard deviation:",
      np.min(train_stds))

print("\nNaN values in scaled training data:",
      np.isnan(X_train_scaled).sum())

print("Infinite values in scaled training data:",
      np.isinf(X_train_scaled).sum())

In [ ]:
# Calculate statistics of scaled test data
test_means = X_test_scaled.mean(axis=0)
test_stds = X_test_scaled.std(axis=0)

print("Average absolute mean:",
      np.mean(np.abs(test_means)))

print("Average standard deviation:",
      np.mean(test_stds))

print("Maximum absolute mean:",
      np.max(np.abs(test_means)))

print("Minimum standard deviation:",
      np.min(test_stds))

print("\nNaN values in scaled test data:",
      np.isnan(X_test_scaled).sum())

print("Infinite values in scaled test data:",
      np.isinf(X_test_scaled).sum())

In [ ]:
print("Number of features in X_train:",
      X_train.shape[1])

print("Number of features known by scaler:",
      scaler.n_features_in_)

print("\nNumber of feature names:",
      len(X.columns))

print("\nFeature count matches:",
      X_train.shape[1] == scaler.n_features_in_ == len(X.columns))

print("\nFirst 10 features:")
for i, feature in enumerate(X.columns[:10], start=1):
    print(f"{i}. {feature}")

print("\nLast 10 features:")
start_number = len(X.columns) - 9

for i, feature in enumerate(X.columns[-10:], start=start_number):
    print(f"{i}. {feature}")

In [ ]:
print("Binary Classification Dataset")
print("=" * 70)

print("Training samples:", X_train_scaled.shape[0])
print("Testing samples:", X_test_scaled.shape[0])
print("Number of features:", X_train_scaled.shape[1])

print("\nTraining target distribution:")
for encoded_value, count in pd.Series(
    y_binary_train_encoded
).value_counts().sort_index().items():
    label = binary_encoder.inverse_transform([encoded_value])[0]
    percentage = count / len(y_binary_train_encoded) * 100
    print(f"{encoded_value} ({label}): {count:,} ({percentage:.2f}%)")

print("\nTesting target distribution:")
for encoded_value, count in pd.Series(
    y_binary_test_encoded
).value_counts().sort_index().items():
    label = binary_encoder.inverse_transform([encoded_value])[0]
    percentage = count / len(y_binary_test_encoded) * 100
    print(f"{encoded_value} ({label}): {count:,} ({percentage:.2f}%)")

In [ ]:
from sklearn.linear_model import LogisticRegression
import time

print("Training Logistic Regression baseline...")
print("=" * 70)

start_time = time.time()

logistic_model = LogisticRegression(
    class_weight="balanced",
    solver="saga",
    max_iter=100,
    random_state=42
)

logistic_model.fit(
    X_train_scaled,
    y_binary_train_encoded
)

training_time = time.time() - start_time

print("\nLogistic Regression training completed.")
print(f"Training time: {training_time:.2f} seconds")
print(f"Iterations used: {logistic_model.n_iter_[0]}")

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("Evaluating Logistic Regression...")
print("=" * 70)

# Predictions
y_binary_pred = logistic_model.predict(X_test_scaled)

# Metrics
accuracy = accuracy_score(y_binary_test_encoded, y_binary_pred)
precision = precision_score(
    y_binary_test_encoded,
    y_binary_pred,
    average="binary",
    pos_label=0
)
recall = recall_score(
    y_binary_test_encoded,
    y_binary_pred,
    average="binary",
    pos_label=0
)
f1 = f1_score(
    y_binary_test_encoded,
    y_binary_pred,
    average="binary",
    pos_label=0
)

print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-Score  : {f1:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_binary_test_encoded, y_binary_pred))

print("\nClassification Report:")
print(
    classification_report(
        y_binary_test_encoded,
        y_binary_pred,
        target_names=["ATTACK", "BENIGN"]
    )
)

In [ ]:
print(classification_report(
    y_binary_test_encoded,
    y_binary_pred,
    target_names=["ATTACK", "BENIGN"]
))

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt

print("Calculating ROC-AUC...")
print("=" * 70)

# ATTACK is encoded as 0
# Create binary target where ATTACK = 1 and BENIGN = 0
y_attack_true = (y_binary_test_encoded == 0).astype(int)

# Probability of ATTACK
y_attack_proba = logistic_model.predict_proba(X_test_scaled)[:, 0]

# Correct ROC-AUC
roc_auc = roc_auc_score(
    y_attack_true,
    y_attack_proba
)

print(f"ROC-AUC Score: {roc_auc:.4f}")

# ROC Curve
fpr, tpr, thresholds = roc_curve(
    y_attack_true,
    y_attack_proba
)

plt.figure(figsize=(8, 6))

plt.plot(
    fpr,
    tpr,
    label=f"Logistic Regression (AUC = {roc_auc:.4f})"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Logistic Regression")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import joblib
from pathlib import Path

# Paths
MODEL_PATH = Path("../models/logistic_regression_binary.pkl")
METRICS_PATH = Path("../results/metrics/logistic_regression_binary_metrics.txt")

# Save model
joblib.dump(logistic_model, MODEL_PATH)

# Save metrics
with open(METRICS_PATH, "w") as f:
    f.write("Logistic Regression - Binary Attack Detection\n")
    f.write("=" * 60 + "\n")
    f.write(f"Accuracy  : {accuracy:.4f}\n")
    f.write(f"Precision : {precision:.4f}\n")
    f.write(f"Recall    : {recall:.4f}\n")
    f.write(f"F1-Score  : {f1:.4f}\n")
    f.write(f"ROC-AUC   : {roc_auc:.4f}\n")

print("Logistic Regression model saved successfully.")
print(f"Model  : {MODEL_PATH}")
print(f"Metrics: {METRICS_PATH}")

In [ ]:
from sklearn.tree import DecisionTreeClassifier
import time

print("Training Decision Tree...")
print("=" * 70)

start_time = time.time()

decision_tree_model = DecisionTreeClassifier(
    class_weight="balanced",
    random_state=42,
    max_depth=20,
    min_samples_split=10,
    min_samples_leaf=5
)

decision_tree_model.fit(
    X_train,
    y_binary_train_encoded
)

training_time = time.time() - start_time

print("\nDecision Tree training completed.")
print(f"Training time: {training_time:.2f} seconds")
print(f"Tree depth: {decision_tree_model.get_depth()}")
print(f"Number of leaves: {decision_tree_model.get_n_leaves()}")

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

print("Evaluating Decision Tree...")
print("=" * 70)

# Predictions
y_dt_pred = decision_tree_model.predict(X_test)

# Probability of ATTACK
y_dt_proba = decision_tree_model.predict_proba(X_test)[:, 0]

# Metrics
dt_accuracy = accuracy_score(y_binary_test_encoded, y_dt_pred)

dt_precision = precision_score(
    y_binary_test_encoded,
    y_dt_pred,
    pos_label=0
)

dt_recall = recall_score(
    y_binary_test_encoded,
    y_dt_pred,
    pos_label=0
)

dt_f1 = f1_score(
    y_binary_test_encoded,
    y_dt_pred,
    pos_label=0
)

# Correct ROC-AUC: ATTACK = 1
y_attack_true = (y_binary_test_encoded == 0).astype(int)

dt_roc_auc = roc_auc_score(
    y_attack_true,
    y_dt_proba
)

print(f"Accuracy  : {dt_accuracy:.4f}")
print(f"Precision : {dt_precision:.4f}")
print(f"Recall    : {dt_recall:.4f}")
print(f"F1-Score  : {dt_f1:.4f}")
print(f"ROC-AUC   : {dt_roc_auc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_binary_test_encoded, y_dt_pred))

print("\nClassification Report:")
print(classification_report(
    y_binary_test_encoded,
    y_dt_pred,
    target_names=["ATTACK", "BENIGN"]
))

In [ ]:
import joblib
from pathlib import Path

MODEL_PATH = Path("../models/decision_tree_binary.pkl")
METRICS_PATH = Path("../results/metrics/decision_tree_binary_metrics.txt")

# Save model
joblib.dump(decision_tree_model, MODEL_PATH)

# Save metrics
with open(METRICS_PATH, "w") as f:
    f.write("Decision Tree - Binary Attack Detection\n")
    f.write("=" * 60 + "\n")
    f.write(f"Accuracy  : {dt_accuracy:.4f}\n")
    f.write(f"Precision : {dt_precision:.4f}\n")
    f.write(f"Recall    : {dt_recall:.4f}\n")
    f.write(f"F1-Score  : {dt_f1:.4f}\n")
    f.write(f"ROC-AUC   : {dt_roc_auc:.4f}\n")

print("Decision Tree model saved successfully.")
print(f"Model  : {MODEL_PATH}")
print(f"Metrics: {METRICS_PATH}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import time

print("Training Random Forest...")
print("=" * 70)

start_time = time.time()

random_forest_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_split=10,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

random_forest_model.fit(
    X_train,
    y_binary_train_encoded
)

training_time = time.time() - start_time

print("\nRandom Forest training completed.")
print(f"Training time: {training_time:.2f} seconds")
print(f"Number of trees: {random_forest_model.n_estimators}")
print(f"Maximum depth: {random_forest_model.max_depth}")

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

print("Evaluating Random Forest...")
print("=" * 70)

# Predictions
y_rf_pred = random_forest_model.predict(X_test)

# Probability of ATTACK
y_rf_proba = random_forest_model.predict_proba(X_test)[:, 0]

# Metrics
rf_accuracy = accuracy_score(
    y_binary_test_encoded,
    y_rf_pred
)

rf_precision = precision_score(
    y_binary_test_encoded,
    y_rf_pred,
    pos_label=0
)

rf_recall = recall_score(
    y_binary_test_encoded,
    y_rf_pred,
    pos_label=0
)

rf_f1 = f1_score(
    y_binary_test_encoded,
    y_rf_pred,
    pos_label=0
)

# Correct ROC-AUC
y_attack_true = (y_binary_test_encoded == 0).astype(int)

rf_roc_auc = roc_auc_score(
    y_attack_true,
    y_rf_proba
)

print(f"Accuracy  : {rf_accuracy:.4f}")
print(f"Precision : {rf_precision:.4f}")
print(f"Recall    : {rf_recall:.4f}")
print(f"F1-Score  : {rf_f1:.4f}")
print(f"ROC-AUC   : {rf_roc_auc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(
    y_binary_test_encoded,
    y_rf_pred
))

print("\nClassification Report:")
print(
    classification_report(
        y_binary_test_encoded,
        y_rf_pred,
        target_names=["ATTACK", "BENIGN"]
    )
)

In [ ]:
import joblib
from pathlib import Path

# Create directories if they don't exist
MODEL_PATH = Path("../models")
METRICS_PATH = Path("../results/metrics")

MODEL_PATH.mkdir(parents=True, exist_ok=True)
METRICS_PATH.mkdir(parents=True, exist_ok=True)

# Save Random Forest model
joblib.dump(
    random_forest_model,
    MODEL_PATH / "random_forest_binary.pkl"
)

# Save metrics
rf_metrics_text = f"""
Random Forest - Binary Attack Detection

Accuracy  : {rf_accuracy:.4f}
Precision : {rf_precision:.4f}
Recall    : {rf_recall:.4f}
F1-Score  : {rf_f1:.4f}
ROC-AUC   : {rf_roc_auc:.4f}

Confusion Matrix:
{confusion_matrix(y_binary_test_encoded, y_rf_pred)}
"""

with open(
    METRICS_PATH / "random_forest_binary_metrics.txt",
    "w"
) as f:
    f.write(rf_metrics_text)

print("Random Forest model saved successfully.")
print("Model:", MODEL_PATH / "random_forest_binary.pkl")
print("Metrics:", METRICS_PATH / "random_forest_binary_metrics.txt")

In [ ]:
import pandas as pd

model_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree",
        "Random Forest"
    ],
    "Accuracy": [
        accuracy,
        dt_accuracy,
        rf_accuracy
    ],
    "Precision": [
        precision,
        dt_precision,
        rf_precision
    ],
    "Recall": [
        recall,
        dt_recall,
        rf_recall
    ],
    "F1-Score": [
        f1,
        dt_f1,
        rf_f1
    ],
    "ROC-AUC": [
        roc_auc,
        dt_roc_auc,
        rf_roc_auc
    ]
})

print("Binary Classification Model Comparison")
print("=" * 80)
display(model_comparison)

In [ ]:
import matplotlib.pyplot as plt

# Set model names as index
plot_df = model_comparison.set_index("Model")

# Plot metrics
plot_df[["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"]].plot(
    kind="bar",
    figsize=(12, 6)
)

plt.title("Binary Classification Model Comparison")
plt.xlabel("Machine Learning Model")
plt.ylabel("Score")
plt.ylim(0.6, 1.02)
plt.xticks(rotation=0)
plt.legend(loc="lower right")
plt.tight_layout()

# Save figure
plt.savefig(
    "../results/figures/binary_model_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Comparison chart saved successfully.")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

# Create confusion matrix display
cm_rf = confusion_matrix(
    y_binary_test_encoded,
    y_rf_pred
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_rf,
    display_labels=["ATTACK", "BENIGN"]
)

fig, ax = plt.subplots(figsize=(7, 6))

disp.plot(
    ax=ax,
    values_format="d"
)

plt.title("Random Forest - Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()

# Save figure
plt.savefig(
    "../results/figures/random_forest_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Random Forest confusion matrix saved successfully.")

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt

# Convert ATTACK into the positive class
y_attack_true = (y_binary_test_encoded == 0).astype(int)

# Probability of ATTACK
y_attack_proba = random_forest_model.predict_proba(X_test)[:, 0]

# Calculate ROC curve
rf_fpr, rf_tpr, rf_thresholds = roc_curve(
    y_attack_true,
    y_attack_proba
)

# Calculate ROC-AUC
rf_roc_auc = roc_auc_score(
    y_attack_true,
    y_attack_proba
)

# Plot ROC curve
plt.figure(figsize=(8, 6))

plt.plot(
    rf_fpr,
    rf_tpr,
    label=f"Random Forest (AUC = {rf_roc_auc:.4f})"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random Classifier"
)

plt.title("Random Forest - ROC Curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")
plt.grid(True)
plt.tight_layout()

# Save figure
plt.savefig(
    "../results/figures/random_forest_roc_curve.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(f"Random Forest ROC-AUC: {rf_roc_auc:.4f}")
print("ROC curve saved successfully.")

In [ ]:
# Save binary classification model comparison

comparison_path = "../results/metrics/binary_model_comparison.csv"

model_comparison.to_csv(
    comparison_path,
    index=False
)

print("Binary model comparison saved successfully.")
print("File:", comparison_path)

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Recreate multiclass label encoder from the current training labels
multiclass_label_encoder = LabelEncoder()

# Fit using the original multiclass training labels
multiclass_label_encoder.fit(y_multiclass_train)

# Encode current train and test labels
y_multiclass_train_encoded = multiclass_label_encoder.transform(
    y_multiclass_train
)

y_multiclass_test_encoded = multiclass_label_encoder.transform(
    y_multiclass_test
)

print("Multiclass label encoder recreated successfully.")
print("Classes:")
print(multiclass_label_encoder.classes_)

In [ ]:
from collections import Counter

print("Multiclass Classification Dataset")
print("=" * 70)

print(f"Training samples : {len(y_multiclass_train_encoded):,}")
print(f"Testing samples  : {len(y_multiclass_test_encoded):,}")
print(f"Number of classes: {len(multiclass_label_encoder.classes_)}")

print("\nClass distribution in training data:")
print("-" * 70)

class_counts = Counter(y_multiclass_train_encoded)

for class_id, class_name in enumerate(multiclass_label_encoder.classes_):
    count = class_counts[class_id]
    percentage = (count / len(y_multiclass_train_encoded)) * 100
    
    print(
        f"{class_id:2d} | "
        f"{class_name:<25} | "
        f"{count:>10,} | "
        f"{percentage:>7.4f}%"
    )

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# Calculate balanced class weights using the CURRENT training data
class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_multiclass_train_encoded),
    y=y_multiclass_train_encoded
)

# Convert to dictionary
multiclass_class_weights = {
    class_id: weight
    for class_id, weight in zip(
        np.unique(y_multiclass_train_encoded),
        class_weights_array
    )
}

print("Multiclass Class Weights")
print("=" * 70)

for class_id, class_name in enumerate(multiclass_label_encoder.classes_):
    print(
        f"{class_id:2d} | "
        f"{class_name:<25} | "
        f"Weight: {multiclass_class_weights[class_id]:.4f}"
    )

In [ ]:
# Create class-weight dictionary using class names
multiclass_class_weights_named = {
    class_name: multiclass_class_weights[class_id]
    for class_id, class_name in enumerate(
        multiclass_label_encoder.classes_
    )
}

print("Multiclass class-weight dictionary created successfully.")
print("=" * 70)

for class_name, weight in multiclass_class_weights_named.items():
    print(f"{class_name:<25} : {weight:.4f}")

In [ ]:
from sklearn.tree import DecisionTreeClassifier
import time

print("Training Multiclass Decision Tree...")
print("=" * 70)

start_time = time.time()

multiclass_decision_tree = DecisionTreeClassifier(
    class_weight=multiclass_class_weights,
    random_state=42,
    max_depth=20,
    min_samples_split=10,
    min_samples_leaf=5
)

multiclass_decision_tree.fit(
    X_train,
    y_multiclass_train_encoded
)

training_time = time.time() - start_time

print("\nMulticlass Decision Tree training completed.")
print(f"Training time : {training_time:.2f} seconds")
print(f"Tree depth    : {multiclass_decision_tree.get_depth()}")
print(f"Number of leaves : {multiclass_decision_tree.get_n_leaves()}")

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("Evaluating Multiclass Decision Tree...")
print("=" * 70)

# Predictions
y_multiclass_dt_pred = multiclass_decision_tree.predict(X_test)

# Overall metrics
dt_multi_accuracy = accuracy_score(
    y_multiclass_test_encoded,
    y_multiclass_dt_pred
)

dt_multi_precision = precision_score(
    y_multiclass_test_encoded,
    y_multiclass_dt_pred,
    average="macro",
    zero_division=0
)

dt_multi_recall = recall_score(
    y_multiclass_test_encoded,
    y_multiclass_dt_pred,
    average="macro",
    zero_division=0
)

dt_multi_f1 = f1_score(
    y_multiclass_test_encoded,
    y_multiclass_dt_pred,
    average="macro",
    zero_division=0
)

dt_multi_weighted_f1 = f1_score(
    y_multiclass_test_encoded,
    y_multiclass_dt_pred,
    average="weighted",
    zero_division=0
)

print(f"Accuracy       : {dt_multi_accuracy:.4f}")
print(f"Macro Precision: {dt_multi_precision:.4f}")
print(f"Macro Recall   : {dt_multi_recall:.4f}")
print(f"Macro F1-Score : {dt_multi_f1:.4f}")
print(f"Weighted F1    : {dt_multi_weighted_f1:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_multiclass_test_encoded,
        y_multiclass_dt_pred,
        target_names=multiclass_label_encoder.classes_,
        zero_division=0
    )
)

In [ ]:
import joblib
from pathlib import Path

MODEL_PATH = Path("../models")
METRICS_PATH = Path("../results/metrics")

MODEL_PATH.mkdir(parents=True, exist_ok=True)
METRICS_PATH.mkdir(parents=True, exist_ok=True)

# Save multiclass Decision Tree
joblib.dump(
    multiclass_decision_tree,
    MODEL_PATH / "decision_tree_multiclass.pkl"
)

# Save metrics
dt_multiclass_metrics_text = f"""
Multiclass Decision Tree

Accuracy        : {dt_multi_accuracy:.4f}
Macro Precision : {dt_multi_precision:.4f}
Macro Recall    : {dt_multi_recall:.4f}
Macro F1-Score  : {dt_multi_f1:.4f}
Weighted F1     : {dt_multi_weighted_f1:.4f}

Classification Report:
{classification_report(
    y_multiclass_test_encoded,
    y_multiclass_dt_pred,
    target_names=multiclass_label_encoder.classes_,
    zero_division=0
)}
"""

with open(
    METRICS_PATH / "decision_tree_multiclass_metrics.txt",
    "w"
) as f:
    f.write(dt_multiclass_metrics_text)

print("Multiclass Decision Tree saved successfully.")
print(
    "Model:",
    MODEL_PATH / "decision_tree_multiclass.pkl"
)
print(
    "Metrics:",
    METRICS_PATH / "decision_tree_multiclass_metrics.txt"
)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

# Create confusion matrix
cm_multiclass_dt = confusion_matrix(
    y_multiclass_test_encoded,
    y_multiclass_dt_pred
)

# Display confusion matrix
fig, ax = plt.subplots(figsize=(14, 12))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_multiclass_dt,
    display_labels=multiclass_label_encoder.classes_
)

disp.plot(
    ax=ax,
    values_format="d",
    xticks_rotation=90
)

plt.title("Multiclass Decision Tree - Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()

# Save figure
plt.savefig(
    "../results/figures/decision_tree_multiclass_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Multiclass confusion matrix saved successfully.")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import time

print("Training Multiclass Random Forest...")

start_time = time.time()

multiclass_random_forest = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_split=10,
    min_samples_leaf=2,
    class_weight=multiclass_class_weights,
    random_state=42,
    n_jobs=-1
)

multiclass_random_forest.fit(
    X_train,
    y_multiclass_train_encoded
)

training_time = time.time() - start_time

print("Multiclass Random Forest training completed.")
print(f"Training time: {training_time:.2f} seconds")
print(f"Number of trees: {multiclass_random_forest.n_estimators}")
print(f"Maximum depth: {multiclass_random_forest.max_depth}")

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

print("Evaluating Multiclass Random Forest...")

# Predictions
y_multiclass_rf_pred = multiclass_random_forest.predict(X_test)

# Metrics
rf_accuracy = accuracy_score(
    y_multiclass_test_encoded,
    y_multiclass_rf_pred
)

rf_precision_macro = precision_score(
    y_multiclass_test_encoded,
    y_multiclass_rf_pred,
    average="macro",
    zero_division=0
)

rf_recall_macro = recall_score(
    y_multiclass_test_encoded,
    y_multiclass_rf_pred,
    average="macro",
    zero_division=0
)

rf_f1_macro = f1_score(
    y_multiclass_test_encoded,
    y_multiclass_rf_pred,
    average="macro",
    zero_division=0
)

rf_f1_weighted = f1_score(
    y_multiclass_test_encoded,
    y_multiclass_rf_pred,
    average="weighted",
    zero_division=0
)

print("\nMulticlass Random Forest Results")
print("--------------------------------")
print(f"Accuracy         : {rf_accuracy:.4f}")
print(f"Macro Precision  : {rf_precision_macro:.4f}")
print(f"Macro Recall     : {rf_recall_macro:.4f}")
print(f"Macro F1         : {rf_f1_macro:.4f}")
print(f"Weighted F1      : {rf_f1_weighted:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_multiclass_test_encoded,
        y_multiclass_rf_pred,
        target_names=multiclass_label_encoder.classes_,
        zero_division=0
    )
)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm_multiclass_rf = confusion_matrix(
    y_multiclass_test_encoded,
    y_multiclass_rf_pred
)

fig, ax = plt.subplots(figsize=(14, 12))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_multiclass_rf,
    display_labels=multiclass_label_encoder.classes_
)

disp.plot(
    ax=ax,
    values_format="d",
    xticks_rotation=90
)

plt.title("Multiclass Random Forest - Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()

plt.savefig(
    "../results/figures/random_forest_multiclass_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Multiclass Random Forest confusion matrix saved successfully.")

In [ ]:
import joblib
from pathlib import Path

# Create required directories
MODEL_PATH = Path("../models")
METRICS_PATH = Path("../results/metrics")

MODEL_PATH.mkdir(parents=True, exist_ok=True)
METRICS_PATH.mkdir(parents=True, exist_ok=True)

# Save the trained Random Forest model
joblib.dump(
    multiclass_random_forest,
    MODEL_PATH / "random_forest_multiclass.pkl"
)

# Save evaluation metrics
with open(
    METRICS_PATH / "random_forest_multiclass_metrics.txt",
    "w"
) as f:
    f.write("Multiclass Random Forest Evaluation\n")
    f.write("=" * 45 + "\n")
    f.write(f"Accuracy        : {rf_accuracy:.4f}\n")
    f.write(f"Macro Precision : {rf_precision_macro:.4f}\n")
    f.write(f"Macro Recall    : {rf_recall_macro:.4f}\n")
    f.write(f"Macro F1        : {rf_f1_macro:.4f}\n")
    f.write(f"Weighted F1     : {rf_f1_weighted:.4f}\n\n")

    f.write("Classification Report\n")
    f.write("=" * 45 + "\n")
    f.write(
        classification_report(
            y_multiclass_test_encoded,
            y_multiclass_rf_pred,
            target_names=multiclass_label_encoder.classes_,
            zero_division=0
        )
    )

print("Multiclass Random Forest model saved successfully.")
print("Metrics saved successfully.")
print("\nModel file:")
print(MODEL_PATH / "random_forest_multiclass.pkl")

print("\nMetrics file:")
print(METRICS_PATH / "random_forest_multiclass_metrics.txt")

In [ ]:
import pandas as pd

multiclass_model_comparison = pd.DataFrame({
    "Model": [
        "Decision Tree",
        "Random Forest"
    ],
    "Accuracy": [
        dt_multi_accuracy,
        rf_accuracy
    ],
    "Macro Precision": [
        dt_multi_precision,
        rf_precision_macro
    ],
    "Macro Recall": [
        dt_multi_recall,
        rf_recall_macro
    ],
    "Macro F1": [
        dt_multi_f1,
        rf_f1_macro
    ],
    "Weighted F1": [
        dt_multi_weighted_f1,
        rf_f1_weighted
    ]
})

print("Multiclass Model Comparison")
print("=" * 70)
print(multiclass_model_comparison.to_string(index=False))

# Save comparison
multiclass_model_comparison.to_csv(
    "../results/metrics/multiclass_model_comparison.csv",
    index=False
)

print("\nComparison saved successfully.")

In [ ]:
import matplotlib.pyplot as plt

metrics_to_plot = [
    "Accuracy",
    "Macro Precision",
    "Macro Recall",
    "Macro F1",
    "Weighted F1"
]

x = range(len(metrics_to_plot))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 7))

dt_values = multiclass_model_comparison.iloc[0][metrics_to_plot].values
rf_values = multiclass_model_comparison.iloc[1][metrics_to_plot].values

ax.bar(
    [i - width / 2 for i in x],
    dt_values,
    width,
    label="Decision Tree"
)

ax.bar(
    [i + width / 2 for i in x],
    rf_values,
    width,
    label="Random Forest"
)

ax.set_xlabel("Evaluation Metrics")
ax.set_ylabel("Score")
ax.set_title("Multiclass Model Comparison")
ax.set_xticks(list(x))
ax.set_xticklabels(metrics_to_plot)
ax.set_ylim(0, 1.05)

ax.legend()

plt.tight_layout()

plt.savefig(
    "../results/figures/multiclass_model_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Multiclass model comparison graph saved successfully.")

In [ ]:
import numpy as np
import pandas as pd

print("Preparing data for anomaly detection...")

# Select only BENIGN traffic from the training set
benign_train_mask = (y_binary_train == "BENIGN")

X_benign_train = X_train.loc[benign_train_mask].copy()

print("\nAnomaly Detection Data")
print("-" * 40)
print(f"Total training samples       : {len(X_train):,}")
print(f"BENIGN training samples      : {len(X_benign_train):,}")
print(f"Number of features           : {X_benign_train.shape[1]}")

# Isolation Forest does not need all 1.67 million benign samples.
# Use a representative random sample for efficient training.
ANOMALY_SAMPLE_SIZE = 100000

if len(X_benign_train) > ANOMALY_SAMPLE_SIZE:
    X_benign_anomaly_train = X_benign_train.sample(
        n=ANOMALY_SAMPLE_SIZE,
        random_state=42
    )
else:
    X_benign_anomaly_train = X_benign_train.copy()

print(f"\nSamples selected for Isolation Forest: "
      f"{len(X_benign_anomaly_train):,}")

print("\nData preparation completed successfully.")

In [ ]:
print("Scaling anomaly detection training data...")

# Use the scaler already fitted on X_train
X_benign_anomaly_train_scaled = scaler.transform(
    X_benign_anomaly_train
)

print("\nScaled anomaly detection data")
print("-" * 40)
print(f"Shape: {X_benign_anomaly_train_scaled.shape}")

print(
    f"NaN values : "
    f"{np.isnan(X_benign_anomaly_train_scaled).sum()}"
)

print(
    f"Inf values : "
    f"{np.isinf(X_benign_anomaly_train_scaled).sum()}"
)

print("\nScaling completed successfully.")

In [ ]:
from sklearn.ensemble import IsolationForest
import time

print("Training Isolation Forest...")

start_time = time.time()

isolation_forest = IsolationForest(
    n_estimators=100,
    contamination="auto",
    random_state=42,
    n_jobs=-1
)

isolation_forest.fit(X_benign_anomaly_train_scaled)

training_time = time.time() - start_time

print("\nIsolation Forest training completed.")
print(f"Training time     : {training_time:.2f} seconds")
print(f"Number of trees   : {isolation_forest.n_estimators}")
print(f"Contamination     : {isolation_forest.contamination}")

In [ ]:
print("Running Isolation Forest on test data...")

# Scale the test data using the same training scaler
X_test_scaled_for_anomaly = scaler.transform(X_test)

# Generate anomaly predictions
# 1  = normal
# -1 = anomaly
anomaly_predictions = isolation_forest.predict(
    X_test_scaled_for_anomaly
)

# Generate anomaly scores
anomaly_scores = isolation_forest.decision_function(
    X_test_scaled_for_anomaly
)

print("\nAnomaly Detection Results")
print("-" * 40)

print(f"Test samples              : {len(X_test):,}")
print(f"Anomaly predictions shape : {anomaly_predictions.shape}")
print(f"Anomaly scores shape      : {anomaly_scores.shape}")

print("\nPrediction counts:")
print(
    pd.Series(anomaly_predictions)
    .value_counts()
    .sort_index()
)

print("\nScore range:")
print(f"Minimum score : {anomaly_scores.min():.6f}")
print(f"Maximum score : {anomaly_scores.max():.6f}")

print("\nAnomaly detection completed successfully.")

In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score
)

print("Evaluating anomaly detection against actual test labels...")

# Actual binary labels
# ATTACK = 1, BENIGN = 0
y_actual_attack = (y_binary_test == "ATTACK").astype(int)

# Convert Isolation Forest output:
# -1 (anomaly) -> 1
#  1 (normal)  -> 0
y_pred_anomaly = (anomaly_predictions == -1).astype(int)

# Metrics
anomaly_precision = precision_score(
    y_actual_attack,
    y_pred_anomaly,
    zero_division=0
)

anomaly_recall = recall_score(
    y_actual_attack,
    y_pred_anomaly,
    zero_division=0
)

anomaly_f1 = f1_score(
    y_actual_attack,
    y_pred_anomaly,
    zero_division=0
)

cm_anomaly = confusion_matrix(
    y_actual_attack,
    y_pred_anomaly
)

print("\nAnomaly Detection Evaluation")
print("-" * 40)
print(f"Precision : {anomaly_precision:.4f}")
print(f"Recall    : {anomaly_recall:.4f}")
print(f"F1 Score  : {anomaly_f1:.4f}")

print("\nConfusion Matrix:")
print(cm_anomaly)

print("\nClassification Report:")
print(
    classification_report(
        y_actual_attack,
        y_pred_anomaly,
        target_names=["BENIGN", "ATTACK"],
        zero_division=0
    )
)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(7, 6))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_anomaly,
    display_labels=["BENIGN", "ATTACK"]
)

disp.plot(
    ax=ax,
    values_format="d"
)

plt.title("Isolation Forest - Anomaly Detection Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()

plt.savefig(
    "../results/figures/isolation_forest_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Isolation Forest confusion matrix saved successfully.")

In [ ]:
import joblib
from pathlib import Path

MODEL_PATH = Path("../models")
METRICS_PATH = Path("../results/metrics")

MODEL_PATH.mkdir(parents=True, exist_ok=True)
METRICS_PATH.mkdir(parents=True, exist_ok=True)

# Save Isolation Forest model
joblib.dump(
    isolation_forest,
    MODEL_PATH / "isolation_forest.pkl"
)

# Save evaluation metrics
with open(
    METRICS_PATH / "isolation_forest_metrics.txt",
    "w"
) as f:
    f.write("Isolation Forest - Anomaly Detection Evaluation\n")
    f.write("=" * 55 + "\n")
    f.write(f"Precision : {anomaly_precision:.4f}\n")
    f.write(f"Recall    : {anomaly_recall:.4f}\n")
    f.write(f"F1 Score  : {anomaly_f1:.4f}\n\n")

    f.write("Confusion Matrix\n")
    f.write("=" * 55 + "\n")
    f.write(str(cm_anomaly))
    f.write("\n\n")

    f.write("Classification Report\n")
    f.write("=" * 55 + "\n")
    f.write(
        classification_report(
            y_actual_attack,
            y_pred_anomaly,
            target_names=["BENIGN", "ATTACK"],
            zero_division=0
        )
    )

print("Isolation Forest model saved successfully.")
print("Metrics saved successfully.")

print("\nModel file:")
print(MODEL_PATH / "isolation_forest.pkl")

print("\nMetrics file:")
print(METRICS_PATH / "isolation_forest_metrics.txt")

In [ ]:
print("Preparing gated predictions (binary RF -> multiclass DT)...")

# Step A: binary gate with the Random Forest (same as the app).
# The Random Forest was trained on the unscaled X_train, so it gets unscaled X_test.
binary_pred_encoded = random_forest_model.predict(X_test)
binary_pred_labels = binary_encoder.inverse_transform(binary_pred_encoded)
attack_mask_test = (binary_pred_labels == "ATTACK")

# Step B: multiclass model runs only on rows flagged as ATTACK.
final_labels = np.array(["BENIGN"] * len(X_test), dtype=object)
final_confidence = np.zeros(len(X_test))

if attack_mask_test.any():
    attack_rows_test = X_test.loc[attack_mask_test]
    attack_probs = multiclass_decision_tree.predict_proba(attack_rows_test)

    # BENIGN can never be the attack label; -1 so argmax picks a real attack class
    benign_code = multiclass_label_encoder.transform(["BENIGN"])[0]
    benign_col = list(multiclass_decision_tree.classes_).index(benign_code)
    attack_probs[:, benign_col] = -1

    best_cols = attack_probs.argmax(axis=1)
    pred_codes = multiclass_decision_tree.classes_[best_cols]

    final_labels[attack_mask_test] = multiclass_label_encoder.inverse_transform(pred_codes)

    # Confidence = probability of the predicted attack class (0 to 1). BENIGN rows stay 0.
    final_confidence[attack_mask_test] = attack_probs.max(axis=1)

predicted_attack_labels = final_labels

threat_events = pd.DataFrame({
    "Predicted_Attack": predicted_attack_labels
})

threat_events["Is_Attack"] = (
    threat_events["Predicted_Attack"] != "BENIGN"
)

print("\nThreat Correlation Data")
print("-" * 40)
print(f"Total test events : {len(threat_events):,}")
print(f"Attack events    : {threat_events['Is_Attack'].sum():,}")
print(f"BENIGN events    : {(~threat_events['Is_Attack']).sum():,}")

print("\nPredicted attack distribution:")
print(
    threat_events.loc[
        threat_events["Is_Attack"],
        "Predicted_Attack"
    ].value_counts()
)

print("\nThreat event preparation completed successfully.")

In [ ]:
print("Creating threat category mapping...")

# Map individual attack types to broader threat categories
threat_category_mapping = {
    "BENIGN": "BENIGN",

    # Reconnaissance
    "PortScan": "Reconnaissance",

    # Denial of Service
    "DDoS": "Denial_of_Service",
    "DoS_Hulk": "Denial_of_Service",
    "DoS_GoldenEye": "Denial_of_Service",
    "DoS_Slowhttptest": "Denial_of_Service",
    "DoS_Slowloris": "Denial_of_Service",

    # Credential / Brute-force attacks
    "FTP_Patator": "Credential_Attack",
    "SSH_Patator": "Credential_Attack",

    # Web attacks
    "WebAttack_BruteForce": "Web_Attack",
    "WebAttack_SQLInjection": "Web_Attack",
    "WebAttack_XSS": "Web_Attack",

    # Other suspicious activity
    "Bot": "Malware_or_Bot",
    "Infiltration": "Infiltration",
    "Heartbleed": "Vulnerability_Exploit"
}

# Add broad threat category
threat_events["Threat_Category"] = (
    threat_events["Predicted_Attack"]
    .map(threat_category_mapping)
    .fillna("Other")
)

print("\nThreat category distribution:")
print(
    threat_events["Threat_Category"]
    .value_counts()
)

print("\nThreat category mapping completed successfully.")

In [ ]:
print("Analyzing threat category co-occurrence...")

# Get all non-benign threat categories
attack_categories = sorted(
    threat_events.loc[
        threat_events["Threat_Category"] != "BENIGN",
        "Threat_Category"
    ].unique()
)

print("\nThreat categories detected:")
for category in attack_categories:
    print("-", category)

# Create binary presence matrix
category_presence = pd.DataFrame(
    0,
    index=["Test Dataset"],
    columns=attack_categories
)

# Mark each category as present
for category in attack_categories:
    category_presence.loc["Test Dataset", category] = (
        threat_events["Threat_Category"]
        .eq(category)
        .sum()
    )

print("\nThreat category event counts:")
print(category_presence.T)

print("\nThreat co-occurrence analysis completed successfully.")

In [ ]:
print("Building threat correlation matrix...")

# Create a binary indicator for each threat category
category_binary = pd.get_dummies(
    threat_events["Threat_Category"]
)

# Remove BENIGN from threat correlation analysis
if "BENIGN" in category_binary.columns:
    category_binary = category_binary.drop(columns=["BENIGN"])

# Calculate correlation between threat categories
threat_correlation_matrix = category_binary.corr()

print("\nThreat Correlation Matrix:")
print(threat_correlation_matrix.round(3))

# Save correlation matrix
threat_correlation_matrix.to_csv(
    "../results/metrics/threat_correlation_matrix.csv"
)

print("\nThreat correlation matrix saved successfully.")

In [ ]:
print("Creating threat profile summary...")

# Count predicted threat categories excluding BENIGN
threat_profile = (
    threat_events.loc[
        threat_events["Threat_Category"] != "BENIGN",
        "Threat_Category"
    ]
    .value_counts()
    .rename("Event_Count")
    .to_frame()
)

# Calculate percentage among all test events
threat_profile["Percentage_of_Test_Data"] = (
    threat_profile["Event_Count"]
    / len(threat_events)
    * 100
)

# Calculate percentage among predicted attack events
total_predicted_attacks = threat_profile["Event_Count"].sum()

threat_profile["Percentage_of_Predicted_Attacks"] = (
    threat_profile["Event_Count"]
    / total_predicted_attacks
    * 100
)

print("\nThreat Profile Summary")
print("=" * 75)
print(threat_profile.round(2))

# Save threat profile
threat_profile.to_csv(
    "../results/metrics/threat_profile_summary.csv"
)

print("\nThreat profile summary saved successfully.")

In [ ]:
import matplotlib.pyplot as plt

plot_data = threat_profile.sort_values(
    "Event_Count",
    ascending=True
)

plt.figure(figsize=(10, 6))

plt.barh(
    plot_data.index,
    plot_data["Event_Count"]
)

plt.xlabel("Number of Predicted Events")
plt.ylabel("Threat Category")
plt.title("CyberShield-AI Threat Profile")

plt.tight_layout()

plt.savefig(
    "../results/figures/threat_profile.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Threat profile visualization saved successfully.")

In [ ]:
import numpy as np
import pandas as pd

print("Preparing risk-scoring inputs...")

# Anomaly detection on scaled test data (Isolation Forest trained earlier)
anomaly_predictions = isolation_forest.predict(X_test_scaled)
anomaly_flag = (anomaly_predictions == -1).astype(int)

# Threat categories and severity (labels come from the gated predictions above)
severity_map = {
    "BENIGN": 0,
    "Reconnaissance": 40,
    "Denial_of_Service": 70,
    "Credential_Attack": 75,
    "Web_Attack": 70,
    "Malware_or_Bot": 85,
    "Infiltration": 95,
    "Vulnerability_Exploit": 90,
    "Other": 50
}

risk_category_map = {
    "BENIGN": "BENIGN",
    "PortScan": "Reconnaissance",
    "DDoS": "Denial_of_Service",
    "DoS_Hulk": "Denial_of_Service",
    "DoS_GoldenEye": "Denial_of_Service",
    "DoS_Slowhttptest": "Denial_of_Service",
    "DoS_Slowloris": "Denial_of_Service",
    "FTP_Patator": "Credential_Attack",
    "SSH_Patator": "Credential_Attack",
    "WebAttack_BruteForce": "Web_Attack",
    "WebAttack_SQLInjection": "Web_Attack",
    "WebAttack_XSS": "Web_Attack",
    "Bot": "Malware_or_Bot",
    "Infiltration": "Infiltration",
    "Heartbleed": "Vulnerability_Exploit"
}

risk_threat_categories = (
    pd.Series(final_labels)
    .map(risk_category_map)
    .fillna("Other")
)

severity_score = risk_threat_categories.map(severity_map)

risk_inputs = pd.DataFrame({
    "Predicted_Attack": final_labels,
    "Threat_Category": risk_threat_categories.values,
    "Attack_Confidence": final_confidence,
    "Anomaly_Flag": anomaly_flag,
    "Severity_Score": severity_score.values
})

print("\nRisk-scoring inputs created successfully.")
print("Risk input shape:", risk_inputs.shape)
print("\nMissing values:")
print(risk_inputs.isna().sum())

In [ ]:
print("Calculating risk scores...")

risk_scores = risk_inputs.copy()

# Anomaly bonus: only for rows the gate classified as ATTACK.
# A BENIGN row always has a bonus of 0.
ANOMALY_BONUS = 15
is_attack_row = (risk_scores["Predicted_Attack"] != "BENIGN").astype(int)

risk_scores["Anomaly_Bonus"] = (
    risk_scores["Anomaly_Flag"] * ANOMALY_BONUS * is_attack_row
)

risk_scores["Risk_Score"] = (
    risk_scores["Severity_Score"]
    + risk_scores["Anomaly_Bonus"]
).clip(0, 100)

def assign_risk_level(score):
    if score <= 25:
        return "Low"
    elif score <= 50:
        return "Medium"
    elif score <= 75:
        return "High"
    else:
        return "Critical"

risk_scores["Risk_Level"] = risk_scores["Risk_Score"].apply(assign_risk_level)

print("\nRisk scoring completed successfully.")
print("\nRisk level distribution:")
print(risk_scores["Risk_Level"].value_counts())
print("\nRisk score statistics:")
print(risk_scores["Risk_Score"].describe())

In [ ]:
print("Saving risk-scoring results...")

# Save complete risk-scoring dataset
risk_scores.to_csv(
    "../results/metrics/risk_scoring_results.csv",
    index=False
)

# Save risk-level distribution
risk_level_distribution = (
    risk_scores["Risk_Level"]
    .value_counts()
    .rename_axis("Risk_Level")
    .reset_index(name="Event_Count")
)

risk_level_distribution["Percentage"] = (
    risk_level_distribution["Event_Count"]
    / len(risk_scores)
    * 100
)

risk_level_distribution.to_csv(
    "../results/metrics/risk_level_distribution.csv",
    index=False
)

# Display verification
print("\nRisk-level distribution:")
print(
    risk_level_distribution.round(2)
)

print("\nRisk-scoring results shape:")
print(risk_scores.shape)

print("\nRisk-scoring results saved successfully.")